# Bootcamp Databricks - Dia 7
## Notebook de apoio: Spark Processing Examples
### Objetivo: testar alguns comando em spark

In [0]:
from pyspark.sql import functions as F

In [0]:
df1 = spark.table('capgemini_academy.gold.fat_vendas')

result = (df1
  .filter(F.col('ven_vl_total_pago') > 0)
  .groupBy('ven_sk_data_venda')
  .agg(F.sum('ven_vl_total_pago').alias('daily_revenue'))
)

In [0]:
df2 = (spark.table('capgemini_academy.bronze.sales')
            .filter(F.col('Order_Date') == '2026-06-07')
            .select('Order_ID','Customer_ID','Order_Value')
            .groupBy('Customer_ID')
            .agg(F.sum(F.col('Order_Value')).alias('total_revenue'))
)

# Executa action
df2.write.mode('overwrite').saveAsTable('capgemini_academy.silver.customer_revenue')

In [0]:
sales = spark.read.table('capgemini_academy.bronze.sales')
customers = spark.read.table('capgemini_academy.bronze.customers')
products = spark.read.table('capgemini_academy.bronze.products')

#sales.printSchema()

In [0]:
sales_clean = (sales
  .filter(F.col('Shipping_Cost') > 0)
  .filter(F.col('Order_Date').isNotNull())
  .select('Order_ID','Customer_ID','Product_ID','Order_Date','Quantity','Unit_Price')
  .withColumn('line_total', F.col('Quantity') * F.col('Unit_Price'))
)

In [0]:
sales_enriched = (sales_clean
  .join(customers, on='Customer_ID', how='left')
  .join(products, on='Product_ID', how='left')
)

In [0]:
daily_revenue = (sales_enriched
  .groupBy('order_date','city','category')
  .agg(
    F.sum('line_total').alias('revenue'),
    F.countDistinct('order_id').alias('orders'),
    F.sum('quantity').alias('units')
  )
)

In [0]:
(daily_revenue.write
  .format('delta')
  .mode('overwrite')
  .option('overwriteSchema', 'true')
  .saveAsTable('capgemini_academy.gold.daily_revenue')
)

In [0]:
daily_revenue.explain(mode='formatted')

# Também útil durante investigação
daily_revenue.explain(mode='extended')

In [0]:
from pyspark.sql import functions as F

catalog = 'capgemini_academy'
schema = 'gold'
source_table = f'{catalog}.{schema}.fat_vendas'
product_table = f'{catalog}.{schema}.dim_produtos'
date_table = f'{catalog}.{schema}.dim_datas'
target_table = f'{catalog}.{schema}.revenue_by_month_category_pyspark'

fato = spark.table(source_table)
produtos = spark.table(product_table)
datas = spark.table(date_table)

In [0]:
result = (fato
  .join(produtos, on=[fato['ven_sk_produto'] == produtos['pro_sk_produto']], how='left')
  .join(datas, on=[fato['ven_sk_data_venda'] == produtos['dat_sk_data']], how='left')
  .groupBy('dat_nu_ano','dat_nu_mes','pro_ds_categoria')
  .agg(
    F.sum('ven_vl_total_pago').alias('revenue'),
    F.countDistinct('ven_id_venda').alias('orders'),
    F.sum('ven_qt_itens').alias('units')
  )
  .withColumnRenamed('dat_nu_ano', 'ano')
  .withColumnRenamed('dat_nu_mes', 'mes')
  .withColumnRenamed('pro_ds_categoria', 'categoria')
  .orderBy('ano', 'mes', F.desc('revenue'))
)

In [0]:
(result.write
  .format('delta')
  .mode('overwrite')
  .option('overwriteSchema','true')
  .saveAsTable(target_table))

In [0]:
spark.table(target_table).count()

In [0]:
spark.table(target_table).filter(F.col('revenue').isNull()).count()

In [0]:
# Execute apenas se quiser reiniciar o exercício do zero
RESET_LAB = True

if RESET_LAB:
    for t in ['capgemini_academy.silver.customer_revenue', 'capgemini_academy.gold.daily_revenue', target_table]:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
        
    print("Laboratório reiniciado.")
else:
    print("RESET_LAB = False. Nenhum objeto foi removido.")

# End